# Classification des Segments Clients — Modele a Risque Minimal (AI Act)

**Risque minimal** — Outil de segmentation marketing interne

## 0. Dependances optionnelles

In [1]:
# Author: Octo Technology MLOps Tribe
%pip install shap pandera --quiet

/Users/louis.delmas/Documents/Stage/model_platform/.venv/bin/python3: No module named pip


Note: you may need to restart the kernel to use updated packages.


## 1. Imports

In [2]:
# Author: Octo Technology MLOps Tribe
import json
import tempfile
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import pandera as pa
from mlflow.models.signature import infer_signature
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, f1_score,
    precision_score, recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
ARTIFACTS_DIR = Path(tempfile.mkdtemp())
print(f"Repertoire artefacts temporaires : {ARTIFACTS_DIR}")

/Users/louis.delmas/Documents/Stage/model_platform/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repertoire artefacts temporaires : /var/folders/7v/nkhkrj4n2cv816zr_kcbm36h0000gn/T/tmpy2o1q9w1


## 2. Configuration MLflow

In [3]:
# Author: Octo Technology MLOps Tribe
PROJECT_NAME = "Credit-Risk-Assessment"
MLFLOW_TRACKING_URI = f"http://model-platform.com/registry/{PROJECT_NAME}/"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("customer_segmentation")
print(f"MLflow tracking URI : {MLFLOW_TRACKING_URI}")

KeyboardInterrupt: 

## 3. Generation des donnees synthetiques

In [ ]:
# Author: Octo Technology MLOps Tribe
np.random.seed(42)
N = 6000

from sklearn.cluster import KMeans as _KMeans
age                      = np.random.randint(20, 75, N)
income                   = np.random.lognormal(mean=10.8, sigma=0.5, size=N).clip(15000, 300000).astype(int)
credit_score             = np.random.randint(300, 850, N)
num_existing_loans       = np.random.randint(0, 10, N)
avg_monthly_balance      = np.random.normal(5000, 8000, N).clip(-5000, 200000).round(2)
num_products             = np.random.randint(1, 12, N)
years_as_customer        = np.random.randint(0, 40, N)
digital_engagement_score = np.random.uniform(0, 10, N).round(2)

X_cluster = np.column_stack([age, income/10000, credit_score/100, avg_monthly_balance/10000, num_products])
kmeans = _KMeans(n_clusters=4, random_state=42, n_init=10)
segment = kmeans.fit_predict(X_cluster)

FEATURES = [
    "age", "income", "credit_score", "num_existing_loans",
    "avg_monthly_balance", "num_products", "years_as_customer", "digital_engagement_score",
]
TARGET = "segment"
PROTECTED_ATTRIBUTES = ["age", "income"]

data_dict = {
    "age": age, "income": income, "credit_score": credit_score,
    "num_existing_loans": num_existing_loans, "avg_monthly_balance": avg_monthly_balance,
    "num_products": num_products, "years_as_customer": years_as_customer,
    "digital_engagement_score": digital_engagement_score, "segment": segment,
}

df = pd.DataFrame(data_dict)
print(f"Dataset : {len(df):,} lignes | Taux cible : {df[TARGET].mean():.1%}")

## 4. Validation Pandera & statistiques descriptives

In [ ]:
# Author: Octo Technology MLOps Tribe
SCHEMA = pa.DataFrameSchema(
    name="customer_segment_input_schema",
    description="Contrat de donnees — Classification des Segments Clients — Modele a Risque Minimal (AI Act)",
    columns={
        "age":                      pa.Column(int,   checks=pa.Check.in_range(18, 100),    nullable=False, description="Age du client — attribut protege"),
        "income":                   pa.Column(int,   checks=pa.Check.in_range(0, 500000),   nullable=False, description="Revenu annuel (EUR) — attribut protege"),
        "credit_score":             pa.Column(int,   checks=pa.Check.in_range(300, 850),    nullable=False, description="Score de credit interne"),
        "num_existing_loans":       pa.Column(int,   checks=pa.Check.in_range(0, 20),       nullable=False, description="Nombre de credits en cours"),
        "avg_monthly_balance":      pa.Column(float, checks=pa.Check.in_range(-10000, 500000), nullable=False, description="Solde moyen mensuel (EUR)"),
        "num_products":             pa.Column(int,   checks=pa.Check.in_range(1, 15),       nullable=False, description="Nombre de produits bancaires"),
        "years_as_customer":        pa.Column(int,   checks=pa.Check.in_range(0, 50),       nullable=False, description="Anciennete client (annees)"),
        "digital_engagement_score": pa.Column(float, checks=pa.Check.in_range(0, 10),       nullable=False, description="Score d'engagement digital (0-10)"),
        "segment":                  pa.Column(int,   checks=pa.Check.in_range(0, 3),        nullable=False, description="Segment client (0-3) — cible"),
    },
    coerce=False,
    strict=True,
)

try:
    SCHEMA.validate(df, lazy=True)
    PANDERA_STATUS = "PASS"
    PANDERA_ERRORS = 0
    print("Validation Pandera : SUCCES")
except pa.errors.SchemaErrors as exc:
    PANDERA_STATUS = "FAIL"
    PANDERA_ERRORS = len(exc.failure_cases)
    print(f"Validation Pandera : ECHEC ({PANDERA_ERRORS} erreurs)")

In [ ]:
# Author: Octo Technology MLOps Tribe
print("=== Statistiques descriptives ===")
display(df[FEATURES + [TARGET]].describe().round(3))

print("\n=== Valeurs manquantes ===")
missing = df.isnull().sum()
print(missing[missing > 0].to_string() if missing.any() else "Aucune valeur manquante.")

print(f"\n=== Distribution de la variable cible ===")
vc = df[TARGET].value_counts().sort_index()
for k, v in vc.items():
    print(f"  Classe {k} : {v:,} ({v/len(df):.1%})")

In [ ]:
# Author: Octo Technology MLOps Tribe
SCHEMA_YAML_EXPORTED = False
try:
    schema_yaml_path = ARTIFACTS_DIR / "pandera_schema.yaml"
    with open(schema_yaml_path, "w") as f:
        f.write(SCHEMA.to_yaml())
    SCHEMA_YAML_EXPORTED = True
    print("Schema Pandera exporte -> pandera_schema.yaml")
except Exception as e:
    print(f"Export YAML non disponible : {e}")

validation_report = {
    "schema_name": SCHEMA.name,
    "validation_status": PANDERA_STATUS,
    "validation_errors": PANDERA_ERRORS,
    "n_rows_validated": int(len(df)),
    "protected_attributes": PROTECTED_ATTRIBUTES,
}
validation_report_path = ARTIFACTS_DIR / "data_validation_report.json"
with open(validation_report_path, "w") as f:
    json.dump(validation_report, f, indent=2, ensure_ascii=False)
print("Rapport de validation exporte -> data_validation_report.json")

## 5. Pretraitement

In [ ]:
# Author: Octo Technology MLOps Tribe
X, y = df[FEATURES], df[TARGET]

X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.15, random_state=42, stratify=y_train_full)

scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=FEATURES)
X_val_sc   = pd.DataFrame(scaler.transform(X_val),       columns=FEATURES)
X_test_sc  = pd.DataFrame(scaler.transform(X_test),      columns=FEATURES)

print(f"Train : {len(X_train):,}  |  Validation : {len(X_val):,}  |  Test : {len(X_test):,}")
print(f"Taux cible — train : {y_train.mean():.1%}  |  test : {y_test.mean():.1%}")

## 6. Entrainement du modele

In [ ]:
# Author: Octo Technology MLOps Tribe
PARAMS = {
    "C":            1.0,
    "max_iter":     500,
    "random_state": 42,
    "class_weight": "balanced",
}

model = LogisticRegression(**PARAMS)
model.fit(X_train_sc, y_train)

val_proba = model.predict_proba(X_val_sc)
if 4 == 2:
    val_auc = roc_auc_score(y_val, val_proba[:, 1])
else:
    val_auc = roc_auc_score(y_val, val_proba, multi_class="ovr", average="macro")
print(f"AUC-ROC validation : {val_auc:.4f}")
print("Entrainement termine.")

## 7. Evaluation sur le jeu de test

In [ ]:
# Author: Octo Technology MLOps Tribe
y_pred  = model.predict(X_test_sc)
y_proba_all = model.predict_proba(X_test_sc)
if 4 == 2:
    y_proba = y_proba_all[:, 1]
    auc_val = round(float(roc_auc_score(y_test, y_proba)), 4)
else:
    y_proba = y_proba_all
    auc_val = round(float(roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")), 4)

METRICS = {
    "accuracy":  round(float(accuracy_score(y_test, y_pred)),  4),
    "precision": round(float(precision_score(y_test, y_pred, average="weighted", zero_division=0)), 4),
    "recall":    round(float(recall_score(y_test, y_pred,    average="weighted", zero_division=0)), 4),
    "f1_score":  round(float(f1_score(y_test, y_pred,        average="weighted", zero_division=0)), 4),
    "auc_roc":   auc_val,
}

report_dict = classification_report(y_test, y_pred, output_dict=True)
print(classification_report(y_test, y_pred))
print("\n".join(f"{k:<12}: {v}" for k, v in METRICS.items()))

In [ ]:
# Author: Octo Technology MLOps Tribe
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
if 4 == 2:
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(fpr, tpr, color="goldenrod", lw=2, label=f"AUC = {METRICS['auc_roc']:.3f}")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.fill_between(fpr, tpr, alpha=0.08, color="goldenrod")
    ax.set_xlabel("Taux de faux positifs (FPR)")
    ax.set_ylabel("Taux de vrais positifs (TPR)")
    ax.set_title("Courbe ROC — Classification Segments Clients")
    ax.legend()
else:
    class_names = ["Segment 0", "Segment 1", "Segment 2", "Segment 3"]
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(7, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, colorbar=True, cmap="Blues")
    ax.set_title("Courbe ROC — Classification Segments Clients")
plt.tight_layout()
roc_path = ARTIFACTS_DIR / "roc_curve.png"
plt.savefig(roc_path, dpi=150)
plt.show()

## 8. Analyse d'equite (Fairness)

In [ ]:
# Author: Octo Technology MLOps Tribe
df_eval = X_test.copy()
df_eval["target"]    = y_test.values
df_eval["predicted"] = y_pred
if 4 == 2:
    df_eval["proba"] = y_proba
else:
    for _i in range(4):
        df_eval[f"proba_{_i}"] = y_proba[:, _i]

def subgroup_metrics(group):
    if len(group) < 20:
        return None
    try:
        if 4 == 2:
            auc = round(float(roc_auc_score(group["target"], group["proba"])), 4) if group["target"].nunique() > 1 else "N/A"
        else:
            proba_cols = [f"proba_{_i}" for _i in range(4)]
            auc = round(float(roc_auc_score(group["target"], group[proba_cols].values, multi_class="ovr", average="macro")), 4) if group["target"].nunique() > 1 else "N/A"
    except Exception:
        auc = "N/A"
    return {
        "n":         int(len(group)),
        "target_rate": round(float(group["target"].mean()), 4),
        "accuracy":  round(float(accuracy_score(group["target"], group["predicted"])), 4),
        "recall":    round(float(recall_score(group["target"], group["predicted"], average="weighted", zero_division=0)), 4),
        "precision": round(float(precision_score(group["target"], group["predicted"], average="weighted", zero_division=0)), 4),
        "auc_roc":   auc,
    }

age_bins = pd.cut(df_eval["age"], bins=[18, 35, 50, 100], labels=["18-35", "36-50", "50+"])
fairness_by_age = {str(k): subgroup_metrics(v) for k, v in df_eval.groupby(age_bins, observed=True)}
print("=== Performance par tranche d'age ===")
print(pd.DataFrame(fairness_by_age).T.to_string())

FAIRNESS_REPORT = {
    "protected_attributes": PROTECTED_ATTRIBUTES,
    "note": "Segmentation marketing — analyse de la distribution des segments par age.",
    "by_age_group": fairness_by_age,
}

fairness_path = ARTIFACTS_DIR / "fairness_report.json"
with open(fairness_path, "w") as f:
    json.dump(FAIRNESS_REPORT, f, indent=2, ensure_ascii=False)
print("Rapport d'equite exporte.")

## 9. Explicabilite

In [ ]:
# Author: Octo Technology MLOps Tribe
if hasattr(model, "feature_importances_"):
    importances = model.feature_importances_
    imp_label = "Importance des variables (Gini)"
elif hasattr(model, "coef_"):
    coef = model.coef_
    importances = np.abs(coef).mean(axis=0) if coef.ndim > 1 else np.abs(coef[0])
    imp_label = "Importance des variables (|coef| moyen)"
else:
    importances = None

if importances is not None:
    fi_df = pd.DataFrame({"feature": FEATURES, "importance": importances}).sort_values("importance", ascending=False)
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ["#c0392b" if f in PROTECTED_ATTRIBUTES else "goldenrod" for f in fi_df["feature"][::-1]]
    ax.barh(fi_df["feature"][::-1], fi_df["importance"][::-1], color=colors)
    ax.set_title(f"{imp_label} — rouge = attribut protege")
    ax.set_xlabel("Importance relative")
    plt.tight_layout()
    fi_path = ARTIFACTS_DIR / "feature_importance.png"
    plt.savefig(fi_path, dpi=150)
    plt.show()
    print(fi_df.to_string(index=False))
else:
    fi_df = None
    print("Aucune methode d'importance disponible pour ce modele.")

In [ ]:
# Author: Octo Technology MLOps Tribe
SHAP_AVAILABLE = False
try:
    import shap
    if hasattr(model, "feature_importances_"):
        explainer = shap.TreeExplainer(model)
    else:
        explainer = shap.Explainer(model, X_train_sc)
    shap_values = explainer.shap_values(X_test_sc.iloc[:300])
    plt.figure()
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values
    shap.summary_plot(sv, X_test_sc.iloc[:300], show=False, plot_size=(10, 6))
    shap_path = ARTIFACTS_DIR / "shap_summary.png"
    plt.savefig(shap_path, dpi=150, bbox_inches="tight")
    plt.close()
    SHAP_AVAILABLE = True
    print("SHAP summary plot genere.")
except ImportError:
    print("SHAP non disponible — installer avec : pip install shap")
except Exception as e:
    print(f"SHAP non disponible pour ce modele : {e}")

## 10. Preparation des artefacts de documentation

In [ ]:
# Author: Octo Technology MLOps Tribe
cr_path = ARTIFACTS_DIR / "classification_report.json"
with open(cr_path, "w") as f:
    json.dump(report_dict, f, indent=2, ensure_ascii=False)

fi_csv_path = ARTIFACTS_DIR / "feature_importance.csv"
fi_df.to_csv(fi_csv_path, index=False)

pp_context = {"schema_name": SCHEMA.name, "pandera_status": PANDERA_STATUS, "pandera_errors": PANDERA_ERRORS}
preprocessing_md = Path("preprocessing_description_segment_template.md").read_text(encoding="utf-8").format_map(pp_context)
pp_path = ARTIFACTS_DIR / "preprocessing_description.md"
pp_path.write_text(preprocessing_md, encoding="utf-8")

print("Artefacts prepares :")
for p in sorted(ARTIFACTS_DIR.iterdir()):
    print(f"  {p.name}")

## 11. Logging MLflow

In [ ]:
# Author: Octo Technology MLOps Tribe
MODEL_NAME  = "customer_segment_classifier"
TEAM        = "mlops-tribe"
ENVIRONMENT = "staging"

model_card_text = Path("model_card_segment.md").read_text(encoding="utf-8")
model_card_path = ARTIFACTS_DIR / "model_card.md"
model_card_path.write_text(model_card_text, encoding="utf-8")

with mlflow.start_run(run_name=f"{MODEL_NAME}_v1") as run:
    mlflow.log_params(PARAMS)
    mlflow.log_param("scaler",           "StandardScaler")
    mlflow.log_param("feature_count",    len(FEATURES))
    mlflow.log_param("features",         ", ".join(FEATURES))
    mlflow.log_param("stratified_split", True)
    mlflow.log_param("train_size",       len(X_train))
    mlflow.log_param("val_size",         len(X_val))
    mlflow.log_param("test_size",        len(X_test))

    mlflow.log_metrics(METRICS)
    mlflow.log_metric("auc_roc_validation", round(val_auc, 4))
    mlflow.log_metric("target_rate_train",  round(float(y_train.mean()), 4))
    mlflow.log_metric("target_rate_test",   round(float(y_test.mean()), 4))
    mlflow.log_metric("dataset_total_size", N)

    mlflow.set_tag("model_type",             "LogisticRegression")
    mlflow.set_tag("framework",              "scikit-learn")
    mlflow.set_tag("data_source",            "Donnees synthetiques bancaires")
    mlflow.set_tag("contains_personal_data", "oui — age et revenu (attributs proteges)")
    mlflow.set_tag("protected_attributes",   "age, income")
    mlflow.set_tag("threshold_accuracy",     "0.75")
    mlflow.set_tag("threshold_f1",           "0.70")
    mlflow.set_tag("threshold_auc_roc",      "0.85")

    mlflow.set_tag("model.author",      "Octo Technology MLOps Tribe")
    mlflow.set_tag("model.team",        TEAM)
    mlflow.set_tag("model.environment", ENVIRONMENT)
    mlflow.set_tag("data.synthetic",    "true")
    mlflow.set_tag("pandera.status",    PANDERA_STATUS)
    mlflow.set_tag("ai_act.risk_level", "minimal")
    mlflow.set_tag("ai_act.annex_ref",  "N/A")
    mlflow.set_tag("ai_act.domain",     "Marketing bancaire — segmentation client")
    mlflow.set_tag("mlflow.note.content", model_card_text)

    for art in [cr_path, fi_csv_path, fi_path, fairness_path, roc_path, pp_path, validation_report_path, model_card_path]:
        mlflow.log_artifact(str(art))
    if SCHEMA_YAML_EXPORTED:
        mlflow.log_artifact(str(schema_yaml_path))
    if SHAP_AVAILABLE:
        mlflow.log_artifact(str(shap_path))

    signature     = infer_signature(X_train_sc, model.predict_proba(X_train_sc))
    input_example = X_train_sc.head(3)
    mlflow.sklearn.log_model(
        sk_model=model,
        name="custom_model",
        registered_model_name=MODEL_NAME,
        signature=signature,
        input_example=input_example,
    )
    run_id = run.info.run_id

print(f"\nRun MLflow : {run_id}")
print(f"   Modele enregistre : {MODEL_NAME}")
print(f"   AUC-ROC : {METRICS['auc_roc']}  |  F1 : {METRICS['f1_score']}  |  Accuracy : {METRICS['accuracy']}")